In [27]:
import pandas as pd


df =pd.read_parquet("integrate/cache/comps_test_blk_2026-02-01.parquet")


print(df.head())


df.to_csv("output_file.csv", index=False)



             rep_lp  rep_age  rep_n_all  bld_lp  bld_ppsf  bld_n  bld_age  \
SYN-0116717     NaN      NaN       0.00   12.72      5.57  20.00     1.25   
SYN-0089197     NaN      NaN       0.00   13.32      6.62  20.00     2.23   
SYN-0146711     NaN      NaN       0.00   13.82      6.77  20.00     4.53   
SYN-0030340     NaN      NaN       0.00   13.25      6.81   3.00    10.97   
SYN-0090901     NaN      NaN       0.00   13.74      6.35  15.00     5.85   

             bld_est  bldbed_lp  comm_lp  comm_ppsf  comm_n  knn_lp  knn_ppsf  \
SYN-0116717    12.37      12.70    12.93       5.84   36.00   12.87      6.05   
SYN-0089197    13.20      13.23    13.29       6.62  100.00   13.42      6.70   
SYN-0146711    14.02      13.88    13.84       6.69   12.00   13.70      6.49   
SYN-0030340    13.18      13.14    13.05       6.55  100.00   12.77      6.38   
SYN-0090901    13.43      13.71    13.72       6.37    9.00   13.47      6.37   

             knn_dist  knn_est   lvl  
SYN-0116717

In [25]:
import numpy as np
import pandas as pd

# 1. Load predictions and ground truth
#preds_path = "integrate/preds/list_aware_comps-lf_kw-True_target-ratio_S25_s0.parquet"
preds_path = "integrate/cache/stage1_best.parquet"
actuals_path = "data/listings.parquet"

df_preds = pd.read_parquet(preds_path)
df_actuals = pd.read_parquet(actuals_path)

# Verify actual price column name (usually 'sold_price' or 'close_price')
target_col = "list_price"

# 2. Merge on listing identifier
df = pd.merge(
    df_preds[["ml_num", "p"]],
    df_actuals[["ml_num", target_col]],
    on="ml_num",
    how="inner",
)
df.rename(columns={"p": "y_pred", target_col: "y_true"}, inplace=True)

# 3. Compute error columns
df["diff"] = df["y_pred"] - df["y_true"]  # Signed difference ($)
df["abs_diff"] = df["diff"].abs()  # Absolute difference ($)
df["squared_error"] = df["diff"] ** 2  # Squared error ($^2)
df["ape_pct"] = (df["abs_diff"] / df["y_true"]) * 100  # Absolute percentage error (%)

# 4. Compute overall metrics
rmse = np.sqrt(df["squared_error"].mean())
mae = df["abs_diff"].mean()
mdape = df["ape_pct"].median()
mape = df["ape_pct"].mean()

print(f"=== Model Evaluation Summary ({len(df):,} samples) ===")
print(f"RMSE:     ${rmse:,.2f}")
print(f"MAE:      ${mae:,.2f}")
print(f"MAPE:     {mape:.2f}%")
print(f"MdAPE:    {mdape:.2f}%\n")

# 5. Summary statistics (Mean, Variance, Std, Min, Max) for each column
cols_to_summarize = ["y_true", "y_pred", "diff", "abs_diff", "ape_pct"]

summary_df = pd.DataFrame(
    {
        "Mean": df[cols_to_summarize].mean(),
        "Variance": df[cols_to_summarize].var(),
        "Std Dev": df[cols_to_summarize].std(),
        "Min": df[cols_to_summarize].min(),
        "25%": df[cols_to_summarize].quantile(0.25),
        "Median (50%)": df[cols_to_summarize].median(),
        "75%": df[cols_to_summarize].quantile(0.75),
        "Max": df[cols_to_summarize].max(),
    }
)

# Format display
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
display(summary_df)

KeyError: "['p'] not in index"

In [21]:
import glob
import os
import sys
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Load Ground Truth Data
# ---------------------------------------------------------
# We use the repository's loader to get the historical data with true 'sold_price'
sys.path.append("shared")
sys.path.append("../shared")
try:
    from harness import load

    df_ground_truth = load()
except Exception:
    # Fallback to direct path if running from integrate/ or root
    gt_path = "data/listings.parquet"
    df_ground_truth = pd.read_parquet(gt_path)

# Ensure sold_price exists and index by ml_num for fast lookup
target_col = "sold_price" if "sold_price" in df_ground_truth.columns else "close_price"
actual_map = df_ground_truth.set_index("ml_num")[target_col].dropna().to_dict()

# ---------------------------------------------------------
# 2. Iterate through all prediction files
# ---------------------------------------------------------
preds_dir = "integrate/preds" if os.path.exists("integrate/preds") else "preds"
parquet_files = glob.glob(os.path.join(preds_dir, "*.parquet"))

print(f"Found {len(parquet_files)} prediction files in '{preds_dir}'...")

results = []

for file_path in parquet_files:
    fname = os.path.basename(file_path)

    try:
        df_p = pd.read_parquet(file_path)

        # Match prediction ml_num with true price
        df_p["y_true"] = df_p["ml_num"].map(actual_map)
        df_valid = df_p.dropna(subset=["y_true", "p"]).copy()

        if len(df_valid) == 0:
            continue

        y_true = df_valid["y_true"].values
        y_pred = df_valid["p"].values

        # Compute Metrics
        diff = y_pred - y_true
        abs_diff = np.abs(diff)
        ape = (abs_diff / y_true) * 100

        rmse = np.sqrt(np.mean(diff**2))
        mape = np.mean(ape)
        mdape = np.median(ape)

        # Extract track and fold from filename for easy grouping
        is_off_market = "off_market" in fname
        track = "off_market" if is_off_market else "list_aware"
        fold = "S25" if "_S25_" in fname else ("F1" if "_F1_" in fname else ("F2" if "_F2_" in fname else "Other"))

        results.append(
            {
                "file": fname,
                "track": track,
                "fold": fold,
                "n_samples": len(df_valid),
                "MdAPE (%)": mdape,
                "MAPE (%)": mape,
                "RMSE ($)": rmse,
            }
        )
    except Exception as e:
        print(f"Skipping {fname} due to error: {e}")

# ---------------------------------------------------------
# 3. Create Leaderboard DataFrame
# ---------------------------------------------------------
df_res = pd.DataFrame(results)

if df_res.empty:
    print("No valid matches found between predictions and ground truth.")
else:
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

    print("\n" + "=" * 80)
    print("🏆 TOP 5 BY MdAPE (Lowest Median % Error - Competition Metric)")
    print("=" * 80)
    display(
        df_res.sort_values(by="MdAPE (%)")
        .head(5)[["file", "track", "fold", "MdAPE (%)", "MAPE (%)", "RMSE ($)"]]
        .reset_index(drop=True)
    )

    print("\n" + "=" * 80)
    print("🏆 TOP 5 BY MAPE (Lowest Mean Absolute % Error)")
    print("=" * 80)
    display(
        df_res.sort_values(by="MAPE (%)")
        .head(5)[["file", "track", "fold", "MAPE (%)", "MdAPE (%)", "RMSE ($)"]]
        .reset_index(drop=True)
    )

    print("\n" + "=" * 80)
    print("🏆 TOP 5 BY RMSE (Lowest Dollar-based Error)")
    print("=" * 80)
    display(
        df_res.sort_values(by="RMSE ($)")
        .head(5)[["file", "track", "fold", "RMSE ($)", "MdAPE (%)", "MAPE (%)"]]
        .reset_index(drop=True)
    )

    # Optional: If you want to see only S25 validation fold
    print("\n" + "=" * 80)
    print("🎯 S25 VALIDATION FOLD LEADERBOARD (Sorted by MdAPE)")
    print("=" * 80)
    s25_df = df_res[df_res["fold"] == "S25"].sort_values(by="MdAPE (%)")
    display(
        s25_df[["file", "track", "MdAPE (%)", "MAPE (%)", "RMSE ($)"]].reset_index(
            drop=True
        )
    )

Found 24 prediction files in 'integrate/preds'...

🏆 TOP 5 BY MdAPE (Lowest Median % Error - Competition Metric)


,file,track,fold,MdAPE (%),MAPE (%),RMSE ($)
0,list_aware_comps-raw_kw-True_target-ratio_F2_s0.parquet,list_aware,F2,3.78,4.79,"99,139.76"
1,list_aware_comps-raw_kw-True_target-ratio_F1_s0.parquet,list_aware,F1,3.90,5.04,"118,401.45"
2,list_aware_comps-lf_kw-True_target-ratio_F1_s0.parquet,list_aware,F1,3.93,5.03,"117,169.49"
3,list_aware_comps-lf_kw-True_target-ratio_F2_s0.parquet,list_aware,F2,3.93,4.95,"100,052.51"
4,list_aware_kw-True_target-ratio_F2_s0.parquet,list_aware,F2,4.00,5.04,"101,247.29"



🏆 TOP 5 BY MAPE (Lowest Mean Absolute % Error)


,file,track,fold,MAPE (%),MdAPE (%),RMSE ($)
0,list_aware_comps-raw_kw-True_target-ratio_F2_s0.parquet,list_aware,F2,4.79,3.78,"99,139.76"
1,list_aware_comps-lf_kw-True_target-ratio_F2_s0.parquet,list_aware,F2,4.95,3.93,"100,052.51"
2,list_aware_comps-lf_kw-True_target-ratio_F1_s0.parquet,list_aware,F1,5.03,3.93,"117,169.49"
3,list_aware_comps-raw_kw-True_target-ratio_F1_s0.parquet,list_aware,F1,5.04,3.90,"118,401.45"
4,list_aware_kw-True_target-ratio_F2_s0.parquet,list_aware,F2,5.04,4.00,"101,247.29"



🏆 TOP 5 BY RMSE (Lowest Dollar-based Error)


,file,track,fold,RMSE ($),MdAPE (%),MAPE (%)
0,list_aware_comps-raw_kw-True_target-ratio_F2_s0.parquet,list_aware,F2,"99,139.76",3.78,4.79
1,list_aware_comps-lf_kw-True_target-ratio_F2_s0.parquet,list_aware,F2,"100,052.51",3.93,4.95
2,list_aware_kw-True_target-ratio_F2_s0.parquet,list_aware,F2,"101,247.29",4.00,5.04
3,list_aware_comps-lf_kw-True_target-ratio_S25_s0.parquet,list_aware,S25,"103,834.44",4.37,5.45
4,list_aware_comps-raw_kw-True_target-ratio_S25_s0.parquet,list_aware,S25,"104,800.00",4.21,5.34



🎯 S25 VALIDATION FOLD LEADERBOARD (Sorted by MdAPE)


,file,track,MdAPE (%),MAPE (%),RMSE ($)
0,list_aware_comps-raw_kw-True_target-ratio_S25_s0.parquet,list_aware,4.21,5.34,"104,800.00"
1,list_aware_comps-lf_kw-True_target-ratio_S25_s0.parquet,list_aware,4.37,5.45,"103,834.44"
2,list_aware_kw-True_target-ratio_S25_s0.parquet,list_aware,4.40,5.50,"106,480.27"
3,list_aware_ref_S25_s0.parquet,list_aware,4.49,5.71,"193,539.73"
4,list_aware_target-ratio_S25_s0.parquet,list_aware,4.52,5.63,"109,824.48"
5,off_market_comps-lf_detrend-True_kw-True_S25_s0.parquet,off_market,7.11,9.33,"256,590.14"
6,off_market_comps-lf_detrend-True_kw-True_S25_s1.parquet,off_market,7.16,9.35,"255,421.42"
7,off_market_comps-lf_detrend-True_kw-True_S25_s2.parquet,off_market,7.16,9.38,"253,910.13"
